# 1) Baixar Arquivo do Google Drive

In [1]:
# Instala o gdown — ferramenta para baixar do Google Drive
!pip install gdown -q

# 2) Baixar o modelo que vai passar pelo processo de Fine Tuning

In [2]:
import gdown
gdown.download(id="1s9M8HgJQ0o4-BCnWQ3imFn_Rhsx8r-Di",
               output="/kaggle/working/number-plate-yolo26s.pt", quiet=False)


Downloading...
From: https://drive.google.com/uc?id=1s9M8HgJQ0o4-BCnWQ3imFn_Rhsx8r-Di
To: /kaggle/working/number-plate-yolo26s.pt
100%|██████████| 20.3M/20.3M [00:00<00:00, 28.0MB/s]


'/kaggle/working/number-plate-yolo26s.pt'

# 3) Baixar o Dataset da UFPR


In [7]:

!gdown --id 1rf4RdXoaGWCPjxfv5BGG-7mFV6otLWLO -O /kaggle/working/yj4Iu2-UFPR-ALPR.zip

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1rf4RdXoaGWCPjxfv5BGG-7mFV6otLWLO
From (redirected): https://drive.google.com/uc?id=1rf4RdXoaGWCPjxfv5BGG-7mFV6otLWLO&confirm=t&uuid=e7c5477a-1c32-48f5-b02a-2fc937e7a8e0
To: /kaggle/working/yj4Iu2-UFPR-ALPR.zip
100%|███████████████████████████████████████| 9.74G/9.74G [01:03<00:00, 152MB/s]


# 4) Extrair Arquivos do Zip

In [8]:
import zipfile
from pathlib import Path

CAMINHO_ZIP          = Path("/kaggle/working/yj4Iu2-UFPR-ALPR.zip")
PASTA_DATASET_KAGGLE = Path("/kaggle/working/UFPR-ALPR dataset")

print("Extraindo...")
with zipfile.ZipFile(CAMINHO_ZIP, "r") as zf:
    zf.extractall(PASTA_DATASET_KAGGLE.parent)
print(f"Pronto: {PASTA_DATASET_KAGGLE}")


Extraindo...
Pronto: /kaggle/working/UFPR-ALPR dataset


# 5) Apagar o .zip

In [10]:
import os
os.remove("/kaggle/working/yj4Iu2-UFPR-ALPR.zip")
print("Zip apagado — espaço liberado.")


Zip apagado — espaço liberado.


# 6) Instalar Ultralytics

In [9]:
!pip install ultralytics -q


# 7) Busca genética de hiperparâmetros (Kaggle): tune_kaggle.py 


In [11]:
import os, shutil
from pathlib import Path
from ultralytics import YOLO

CAMINHO_MODELO       = Path("/kaggle/working/number-plate-yolo26s.pt")
PASTA_DATASET_KAGGLE = Path("/kaggle/working/UFPR-ALPR dataset")
PASTA_YOLO           = Path("/kaggle/working/dataset")
CAMINHO_YAML         = Path("/kaggle/working/dataset.yaml")
PASTA_RESULTADOS     = "/kaggle/working/runs/tune"
NOME_EXPERIMENTO     = "busca_hiperparametros_v1"
EPOCHS_POR_CANDIDATO = 3
ITERACOES            = 20

ESPACO_DE_BUSCA = {
    "lr0": (1e-4, 5e-3), "lrf": (0.001, 0.1), "momentum": (0.8, 0.98),
    "weight_decay": (0.0001, 0.001), "box": (5.0, 15.0), "dfl": (0.5, 3.0),
    "degrees": (0.0, 20.0), "perspective": (0.0, 0.001), "scale": (0.3, 0.7),
    "mosaic": (0.5, 1.0), "fliplr": (0.0, 0.5), "hsv_v": (0.2, 0.6),
}

# Remove estrutura incompleta
shutil.rmtree(PASTA_YOLO, ignore_errors=True)

# Cria estrutura YOLO com symlinks (sem copiar dados)
LARGURA, ALTURA = 1920, 1080
splits = {"training": "train", "validation": "val", "testing": "test"}
for split_yolo in splits.values():
    (PASTA_YOLO / "images" / split_yolo).mkdir(parents=True, exist_ok=True)
    (PASTA_YOLO / "labels" / split_yolo).mkdir(parents=True, exist_ok=True)

for split_ufpr, split_yolo in splits.items():
    pasta_imgs = PASTA_YOLO / "images" / split_yolo
    pasta_lbls = PASTA_YOLO / "labels" / split_yolo
    contagem = 0
    for pasta_track in sorted((PASTA_DATASET_KAGGLE / split_ufpr).iterdir()):
        if not pasta_track.is_dir():
            continue
        for png in sorted(pasta_track.glob("*.png")):
            txt = png.with_suffix(".txt")
            if not txt.exists():
                continue
            os.symlink(png, pasta_imgs / png.name)
            with open(txt, encoding="utf-8") as f:
                for linha in f:
                    if linha.startswith("corners:"):
                        pares = linha.replace("corners:", "").strip().split()
                        xs = [int(p.split(",")[0]) for p in pares]
                        ys = [int(p.split(",")[1]) for p in pares]
                        cx = (min(xs) + max(xs)) / 2 / LARGURA
                        cy = (min(ys) + max(ys)) / 2 / ALTURA
                        w  = (max(xs) - min(xs)) / LARGURA
                        h  = (max(ys) - min(ys)) / ALTURA
                        label = f"0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n"
                        break
            with open(pasta_lbls / png.with_suffix(".txt").name, "w") as f:
                f.write(label)
            contagem += 1
    print(f"  [{split_ufpr:>10}] → {contagem} imagens.")

CAMINHO_YAML.write_text(f"""path: {PASTA_YOLO}
train: images/train
val:   images/val
test:  images/test
nc: 1
names:
  0: license_plate
""", encoding="utf-8")
print("dataset.yaml gerado.\n")

modelo = YOLO(str(CAMINHO_MODELO))
modelo.tune(
    data=str(CAMINHO_YAML),
    epochs=EPOCHS_POR_CANDIDATO,
    iterations=ITERACOES,
    imgsz=1280,
    batch=8,
    optimizer="AdamW",
    freeze=10,
    space=ESPACO_DE_BUSCA,
    project=PASTA_RESULTADOS,
    name=NOME_EXPERIMENTO,
    exist_ok=True,
    plots=True,
    save=False,
    val=True,
)


  [  training] → 1800 imagens.
  [validation] → 900 imagens.
  [   testing] → 1800 imagens.
dataset.yaml gerado.

Tuner: Initialized Tuner instance with 'tune_dir=/kaggle/working/runs/tune/busca_hiperparametros_v1'
Tuner: 💡 Learn about tuning at https://docs.ultralytics.com/guides/hyperparameter-tuning
Tuner: Starting iteration 1/20 with hyperparameters: {'lr0': 0.005, 'lrf': 0.01, 'momentum': 0.937, 'weight_decay': 0.0005, 'box': 7.5, 'dfl': 1.5, 'degrees': 0.0, 'perspective': 0.0, 'scale': 0.5, 'mosaic': 1.0, 'fliplr': 0.5, 'hsv_v': 0.4}
Ultralytics 8.4.47 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/dataset.yaml, degrees=0.0, deterministic=True, device=None, 

# 8) Verificação de Ambiente

In [12]:
from pathlib import Path

print("Modelo :", "✅" if Path("/kaggle/working/number-plate-yolo26s.pt").exists() else "❌")
print("Dataset:", "✅" if Path("/kaggle/working/dataset").exists() else "❌")
print("YAML   :", "✅" if Path("/kaggle/working/dataset.yaml").exists() else "❌")


Modelo : ✅
Dataset: ✅
YAML   : ✅


# 9) Fine-tuning com os melhores hiperparâmetros

In [13]:
from pathlib import Path
from ultralytics import YOLO

modelo = YOLO("/kaggle/working/number-plate-yolo26s.pt")

modelo.train(
    data="/kaggle/working/dataset.yaml",
    epochs=50,
    patience=15,
    imgsz=1280,
    batch=8,
    optimizer="AdamW",
    freeze=10,
    cos_lr=True,
    lr0=0.00285,
    lrf=0.01446,
    momentum=0.94248,
    weight_decay=0.0001,
    box=7.51169,
    dfl=1.22798,
    degrees=0.00545,
    perspective=0.0,
    scale=0.49496,
    mosaic=0.61158,
    fliplr=0.5,
    hsv_v=0.40706,
    project="/kaggle/working/runs/finetune",
    name="best_placas_v2",
    exist_ok=True,
    plots=True,
    save=True,
    val=True,
)


Ultralytics 8.4.47 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.51169, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/dataset.yaml, degrees=0.00545, deterministic=True, device=None, dfl=1.22798, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.40706, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.00285, lrf=0.01446, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/working/number-plate-yolo26s.pt, momentum=0.94248, mosaic=0.61158, multi_scale=0.0, name=best_placas_v2, nbs=64, nms=False, opset=None, opt

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7fc69c11eb40>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 